In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import pandas as pd

In [2]:
data_path = './data/preprocessed/'
# Load preprocessed data
train_dataset = torch.load(data_path+'train_dataset.pth', weights_only=False)
test_dataset = torch.load(data_path+'test_dataset.pth', weights_only=False)
# label_encoder = joblib.load(data_path+'label_encoder.pkl')

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# print(f"Type: {type(train_dataset)}")
# print(f"Length: {len(train_dataset)}")

# # Check what one sample looks like
# sample_X, sample_y = train_dataset[0]
# print(f"\nSample 0:")
# print(f"  X shape: {sample_X.shape}")  # Should be (50, 384)
# print(f"  y shape: {sample_y.shape}")  # Should be (2,)
# print(f"  y values: {sample_y}")       # One-hot encoded [1,0] or [0,1]

In [3]:
# Get all samples from train_dataset
X_train_all, y_train_all = train_dataset[:]  # or iterate through

# Assuming y is one-hot encoded: [1,0] for Success, [0,1] for Anomaly
# Find indices of success cases
success_indices = (y_train_all.argmax(dim=1) == 1).nonzero().squeeze()

# Extract only success cases
X_success = X_train_all[success_indices]
y_success = y_train_all[success_indices]

In [ ]:
# print(len(train_dataset))
# print(X_success.shape)
# print(y_success.shape)

In [5]:
class NextEventDataset(torch.utils.data.Dataset):
    def __init__(self, X, max_len=50):
        self.X = X
        self.max_len = max_len
        self.indices = []

        for seq_idx, seq in enumerate(X):
            seq_len = (seq.sum(dim=-1) != 0).sum().item()
            for t in range(1, seq_len):   # use all pairs
                self.indices.append((seq_idx, t))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        seq_idx, t = self.indices[idx]
        seq = self.X[seq_idx]

        prefix = seq[:t]
        target = seq[t]

        padded = torch.zeros((self.max_len, seq.size(-1)))
        padded[:t] = prefix  # right-padding

        return padded, target


next_event_dataset = NextEventDataset(X_success)
next_event_loader = DataLoader(next_event_dataset, batch_size=64, shuffle=True)

In [6]:
print(f"Dataset size: {len(next_event_dataset)} samples")
print(f"First few indices: {next_event_dataset.indices[:10]}")

# Check a few samples
for i in range(3):
    input_seq, target = next_event_dataset[i]
    print(f"\nSample {i}:")
    print(f"  Input shape: {input_seq.shape}")
    print(f"  Target shape: {target.shape}")
    
    # Check actual content
    seq_len = (input_seq.sum(dim=-1) != 0).sum().item()
    print(f"  Actual sequence length: {seq_len}")
    print(f"  Input - min: {input_seq.min():.4f}, max: {input_seq.max():.4f}")
    print(f"  Target - min: {target.min():.4f}, max: {target.max():.4f}")

Dataset size: 290772 samples
First few indices: [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10)]

Sample 0:
  Input shape: torch.Size([50, 384])
  Target shape: torch.Size([384])
  Actual sequence length: 0
  Input - min: 0.0000, max: 0.0000
  Target - min: 0.0000, max: 0.0000

Sample 1:
  Input shape: torch.Size([50, 384])
  Target shape: torch.Size([384])
  Actual sequence length: 0
  Input - min: 0.0000, max: 0.0000
  Target - min: 0.0000, max: 0.0000

Sample 2:
  Input shape: torch.Size([50, 384])
  Target shape: torch.Size([384])
  Actual sequence length: 0
  Input - min: 0.0000, max: 0.0000
  Target - min: 0.0000, max: 0.0000


In [7]:
import torch
import torch.nn as nn

class NextEventTransformer(nn.Module):
    def __init__(self, embed_dim=384, num_heads=8, num_layers=6,
                 max_seq_len=50, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        # learned positional embeddings
        self.pos_emb = nn.Parameter(
            torch.randn(1, max_seq_len, embed_dim) * 0.02
        )

        # encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            layer_norm_eps=1e-6
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2 * embed_dim),
            nn.GELU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for p in self.encoder.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, T, D)
        pad_mask = (x.sum(dim=-1) == 0)
        # add positional embeddings
        x = x + self.pos_emb[:, :x.size(1)]
        x = x * (self.embed_dim ** -0.5)

        # transformer
        enc = self.encoder(x, src_key_padding_mask=pad_mask)

        # last valid index per sample
        lengths = (~pad_mask).sum(dim=1)
        idx = torch.clamp(lengths - 1, min=0)

        # gather
        b = x.size(0)
        last_h = enc[torch.arange(b), idx]

        # predict next embedding
        return self.head(last_h)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NextEventTransformer().to(device)

In [11]:
print(torch.device)
print(next(model.parameters()).device)

print(torch.cuda.is_available())
print(torch.cuda.device_count())


<class 'torch.device'>
cuda:0
True
1


In [12]:
import torch
import torch.nn.functional as F
import os
from tqdm import tqdm
import math

total_training_steps = len(next_event_loader) * 5

model = NextEventTransformer()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

save_dir = "models/transformer/checkpoints"
os.makedirs(save_dir, exist_ok=True)

global_step = 0
save_every = 500  # steps

# Warmup configuration
warmup_steps = 500
base_lr = 5e-5

import torch
import torch.nn.functional as F
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NextEventTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

warmup_steps = 500
base_lr = 5e-5
total_steps = len(next_event_loader) * 5
global_step = 0

for epoch in range(5):
    model.train()
    pbar = tqdm(next_event_loader, desc=f"Epoch {epoch+1}", ncols=100)

    for batch_X, batch_y in pbar:
        global_step += 1

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # ---- LR schedule ----
        if global_step < warmup_steps:
            lr = base_lr * (global_step / warmup_steps)
        else:
            progress = (global_step - warmup_steps) / (total_steps - warmup_steps)
            progress = min(progress, 1.0)
            lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))

        for g in optimizer.param_groups:
            g["lr"] = lr

        # ---- forward ----
        pred = model(batch_X)
        pred_n = F.normalize(pred, dim=-1)
        tgt_n = F.normalize(batch_y, dim=-1)

        loss = 1 - (pred_n * tgt_n).sum(-1).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": lr})


Epoch 4:  62%|███████████████▍         | 2812/4544 [05:00<03:04,  9.37it/s, loss=0.7912, lr=9.21e-6]


KeyboardInterrupt: 

In [ ]:
# import torch
# import torch.nn.functional as F
# import os
# import math
# from tqdm import tqdm

# # Load your checkpoint
# checkpoint_path = "models/transformer/checkpoints/epoch_0.pt"
# checkpoint = torch.load(checkpoint_path)

# # Initialize model
# model = NextEventTransformer()
# model.load_state_dict(checkpoint["model_state"])

# # Initialize optimizer WITH NEW LR
# optimizer = torch.optim.AdamW(
#     model.parameters(), 
#     lr=3e-5,  # LOWER than before! Start from 3e-5 instead of 5e-5
#     weight_decay=0.01
# )

# # Load optimizer state BUT override the LR
# optimizer.load_state_dict(checkpoint["optimizer_state"])
# for param_group in optimizer.param_groups:
#     param_group['lr'] = 3e-5  # Force the new LR

# save_dir = "models/transformer/checkpoints"
# os.makedirs(save_dir, exist_ok=True)

# # Start from where you left off
# global_step = checkpoint["step"]  # Should be 535
# start_epoch = checkpoint["epoch"] + 1  # Start from epoch 1

# save_every = 500

# # NEW training schedule
# warmup_steps = 500  # Keep same, but we're past it now
# base_lr = 3e-5  # Lower base
# total_training_steps = len(next_event_loader) * 4  # 4 more epochs

# # Only run remaining epochs
# for epoch in range(start_epoch, 5):
#     pbar = tqdm(next_event_loader, desc=f'Epoch {epoch+1}/5')
    
#     for batch_X, batch_y in pbar:
#         global_step += 1

#         # Since we're past warmup, go straight to cosine decay
#         decay_steps = total_training_steps  # Total steps remaining
#         progress = global_step / decay_steps
#         progress = min(progress, 1.0)
#         cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
#         current_lr = base_lr * cosine_decay
        
#         for param_group in optimizer.param_groups:
#             param_group['lr'] = current_lr

#         predictions = model(batch_X)
#         cos = 1 - F.cosine_similarity(predictions, batch_y).mean()
#         mse = F.mse_loss(predictions, batch_y)
#         loss = cos + 0.02 * mse  # Reduced MSE weight

#         optimizer.zero_grad()
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.25)
#         optimizer.step()

#         pbar.set_postfix({
#             'loss': f'{loss.item():.4f}',
#             'lr': f'{current_lr:.2e}',
#             'step': global_step
#         })
        
#         if global_step % 50 == 0:
#             print(f"step {global_step}  loss {loss.item():.4f}  lr {current_lr:.2e}")
        
#         if global_step % save_every == 0:
#             torch.save({
#                 "epoch": epoch,
#                 "step": global_step,
#                 "model_state": model.state_dict(),
#                 "optimizer_state": optimizer.state_dict(),
#                 "loss": loss.item(),
#                 "lr": current_lr
#             }, f"{save_dir}/step_{global_step}.pt")
    
#     pbar.close()
    
#     torch.save({
#         "epoch": epoch,
#         "step": global_step,
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "loss": loss.item(),
#         "lr": current_lr
#     }, f"{save_dir}/epoch_{epoch}_continued.pt")  # Different name to avoid overwriting

#     print(f"\nepoch {epoch} finished  loss {loss.item():.4f}  lr {current_lr:.2e}")
#     print("-" * 60)

Epoch 2/5:   3%|▎         | 15/535 [00:29<16:33,  1.91s/it, loss=0.5723, lr=2.54e-05, step=550]

step 550  loss 0.5723  lr 2.54e-05


Epoch 2/5:  12%|█▏        | 65/535 [01:59<13:58,  1.78s/it, loss=0.5161, lr=2.45e-05, step=600]

step 600  loss 0.5161  lr 2.45e-05


Epoch 2/5:  21%|██▏       | 115/535 [03:30<12:22,  1.77s/it, loss=0.5765, lr=2.37e-05, step=650]

...

step 750  loss 0.5040  lr 2.18e-05


Epoch 2/5:  50%|████▉     | 265/535 [08:16<08:31,  1.90s/it, loss=0.5882, lr=2.08e-05, step=800]

step 800  loss 0.5882  lr 2.08e-05


Epoch 2/5:  53%|█████▎    | 286/535 [08:57<07:48,  1.88s/it, loss=0.6541, lr=2.04e-05, step=821]


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), './models/transformer/transformer.pth')

In [ ]:
# load model

In [ ]:
# from tqdm import tqdm

# model.eval()
# total_loss = 0
# with torch.no_grad():
#     for batch_X, batch_y in tqdm(next_event_loader, desc="Evaluating"):
#         predictions = model(batch_X)
#         loss = criterion(predictions, batch_y)
#         total_loss += loss.item()

# avg_loss = total_loss / len(next_event_loader)
# print(f"Average Prediction MSE: {avg_loss:.6f}")